ล่าสุด

In [ ]:
#%pip install plotly pandas numpy requests pvlib

import pandas as pd
import numpy as np
import requests
from datetime import datetime, timedelta
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pvlib.location import Location
from pvlib.pvsystem import PVSystem
from pvlib.modelchain import ModelChain
from pvlib.temperature import TEMPERATURE_MODEL_PARAMETERS
from pvlib.irradiance import erbs, get_extra_radiation
import pvlib

# ────────── SYSTEM CONFIGURATION ───────────
LAT, LON = 13.850929802369471, 100.55747605767155 # PEA
#LAT, LON = 11.474729262132879, 99.59689042300059 #ทับสะแก
ALTITUDE = 50  # meters
SYSTEM_KW_DC = 5000  # 5 MW
TILT, AZIMUTH = 15, 180
DATE = "2025-07-16"

# System Losses (%)
LOSSES = {
    'soiling': 2.0,
    'shading': 2.0,
    'mismatch': 2.0,
    'wiring': 2.0,
    'connections': 2.0,
    'lid': 2.0,
    'nameplate': 2.0,
    'availability': 2.0,
    'degradation': 3.0  # 0.5% per year * years
}

# ────────── WEATHER DATA FUNCTIONS ───────────
def get_weather_url(day):
    today = datetime.now().date()
    target = datetime.strptime(day, "%Y-%m-%d").date()
    
    if target < today - timedelta(days=1):
        base = "https://archive-api.open-meteo.com/v1/archive"
        tz = ""
    else:
        base = "https://api.open-meteo.com/v1/forecast"
        tz = "&timezone=Asia%2FBangkok"
    
    params = [
        "temperature_2m",
        "relative_humidity_2m",
        "cloud_cover",
        "direct_normal_irradiance",
        "diffuse_radiation",
        "wind_speed_10m"
    ]
    
    return f"{base}?latitude={LAT}&longitude={LON}&hourly={','.join(params)}&start_date={day}&end_date={day}{tz}"

def get_weather(day):
    url = get_weather_url(day)
    resp = requests.get(url, timeout=20)
    data = resp.json()
    
    if "hourly" not in data:
        raise RuntimeError(f"No data for {day}")
    
    h = data["hourly"]
    times = pd.to_datetime(h["time"])
    
    # Timezone handling
    if "archive" in url:
        times = times.tz_localize("UTC").tz_convert("Asia/Bangkok")
    else:
        times = times.tz_localize("Asia/Bangkok")
    
    # Create weather dataframe
    wx = pd.DataFrame({
        'temp_air': h["temperature_2m"],
        'relative_humidity': h.get("relative_humidity_2m", [50]*24),
        'cloud_cover': h.get("cloud_cover", [0]*24),
        'dni': h["direct_normal_irradiance"],
        'dhi': h.get("diffuse_radiation", [0]*24),
        'wind_speed': h["wind_speed_10m"]
    }, index=times)
    
    # Calculate GHI and solar position
    location = Location(LAT, LON, altitude=ALTITUDE)
    solar_pos = location.get_solarposition(times)
    zenith = solar_pos['zenith']
    
    # Calculate GHI
    wx['ghi'] = wx['dni'] * np.cos(np.radians(zenith)) + wx['dhi']
    wx['ghi'] = wx['ghi'].clip(lower=0)
    
    # If DHI missing, use decomposition
    if wx['dhi'].sum() == 0:
        decomp = erbs(wx['ghi'], zenith, times)
        wx['dhi'] = decomp['dhi']
        wx['dni'] = decomp['dni']
    
    return wx

# ────────── PV SYSTEM MODELING ───────────
def calculate_total_loss():
    efficiency = 1.0
    for loss in LOSSES.values():
        efficiency *= (1 - loss/100)
    return (1 - efficiency) * 100

def simulate_pv(wx):
    # Create PV system
    system = PVSystem(
        surface_tilt=TILT,
        surface_azimuth=AZIMUTH,
        module_parameters={
            'pdc0': SYSTEM_KW_DC,
            'gamma_pdc': -0.0045
        },
        inverter_parameters={
            'pdc0': SYSTEM_KW_DC * 1.1,
            'eta_inv_nom': 0.98
        },
        temperature_model_parameters=TEMPERATURE_MODEL_PARAMETERS['sapm']['open_rack_glass_glass']
    )
    
    # Create location and model
    location = Location(LAT, LON, altitude=ALTITUDE)
    mc = ModelChain(
        system, 
        location,
        dc_model='pvwatts',
        ac_model='pvwatts',
        aoi_model='physical',
        spectral_model='no_loss',
        temperature_model='sapm'
    )
    
    # Run simulation
    mc.run_model(wx)
    
    # Apply losses
    total_loss = calculate_total_loss()
    ac_net = mc.results.ac * (1 - total_loss/100)
    
    # Create results
    results = pd.DataFrame({
        'dc_power': mc.results.dc.fillna(0),
        'ac_power_raw': mc.results.ac.fillna(0),
        'ac_power_net': ac_net.fillna(0),
        'cell_temp': mc.results.cell_temperature.fillna(wx['temp_air']),
        'poa_global': mc.results.total_irrad.get('poa_global', pd.Series(0, index=wx.index))
    }, index=wx.index)
    
    return results

# ────────── VISUALIZATION ───────────
def create_plot(wx, results, date):
    # Create figure with subplots
    fig = make_subplots(
        rows=3, cols=1,
        subplot_titles=(
            'Power Output (kW)', 
            'Solar Irradiance (W/m²)', 
            'Temperature (°C)'
        ),
        shared_xaxes=True,
        vertical_spacing=0.08,
        row_heights=[0.5, 0.25, 0.25]
    )
    
    # Time labels
    time_labels = results.index.strftime("%H:%M")
    
    # Plot 1: Power Output
    fig.add_trace(
        go.Scatter(
            x=time_labels,
            y=results['ac_power_net'],
            name='AC Power (Net)',
            line=dict(color='darkblue', width=3),
            fill='tozeroy',
            hovertemplate='Net Power: %{y:,.0f} kW<extra></extra>'
        ), 
        row=1, col=1
    )
    
    fig.add_trace(
        go.Scatter(
            x=time_labels,
            y=results['ac_power_raw'],
            name='AC Power (Before Losses)',
            line=dict(color='lightblue', dash='dash'),
            hovertemplate='Raw Power: %{y:,.0f} kW<extra></extra>'
        ), 
        row=1, col=1
    )
    
    # Plot 2: Irradiance
    fig.add_trace(
        go.Scatter(
            x=time_labels,
            y=wx['ghi'],
            name='GHI',
            line=dict(color='orange'),
            hovertemplate='GHI: %{y:.0f} W/m²<extra></extra>'
        ), 
        row=2, col=1
    )
    
    fig.add_trace(
        go.Scatter(
            x=time_labels,
            y=wx['dni'],
            name='DNI',
            line=dict(color='red'),
            hovertemplate='DNI: %{y:.0f} W/m²<extra></extra>'
        ), 
        row=2, col=1
    )
    
    # Plot 3: Temperature
    fig.add_trace(
        go.Scatter(
            x=time_labels,
            y=wx['temp_air'],
            name='Air Temp',
            line=dict(color='green'),
            hovertemplate='Air: %{y:.1f}°C<extra></extra>'
        ), 
        row=3, col=1
    )
    
    fig.add_trace(
        go.Scatter(
            x=time_labels,
            y=results['cell_temp'],
            name='Cell Temp',
            line=dict(color='darkgreen', dash='dash'),
            hovertemplate='Cell: %{y:.1f}°C<extra></extra>'
        ), 
        row=3, col=1
    )
    
    # Update layout
    total_loss = calculate_total_loss()
    daily_energy = results['ac_power_net'].sum()
    peak_power = results['ac_power_net'].max()
    capacity_factor = (daily_energy / (SYSTEM_KW_DC * 24)) * 100
    
    fig.update_layout(
        title={
            'text': f"Solar Power Forecast - {SYSTEM_KW_DC/1000:.1f} MW System<br>" + 
                   f"<span style='font-size:14px'>Date: {date} | " +
                   f"Daily Energy: {daily_energy/1000:,.1f} KWh | " +
                   f"Peak: {peak_power:,.0f} kW | " +
                   f"CF: {capacity_factor:.1f}% | " +
                   f"Losses: {total_loss:.1f}%</span>",
            'x': 0.5,
            'xanchor': 'center'
        },
        xaxis3_title="Time (Hour)",
        yaxis_title="Power (kW)",
        yaxis2_title="Irradiance (W/m²)",
        yaxis3_title="Temperature (°C)",
        height=800,
        showlegend=True,
        hovermode='x unified'
    )
    
    # Update axes
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
    
    return fig

# ────────── MAIN EXECUTION ───────────
def main():
    print(f"🌞 Solar Forecast for {DATE}")
    print(f"📍 Location: {LAT:.4f}, {LON:.4f}")
    print(f"⚡ System Size: {SYSTEM_KW_DC/1000:.1f} MW")
    
    # Get weather data
    print("\n📊 Fetching weather data...")
    wx = get_weather(DATE)
    
    # Run PV simulation
    print("🔧 Running PV simulation...")
    results = simulate_pv(wx)
    
    # Calculate metrics
    daily_energy = results['ac_power_net'].sum()
    peak_power = results['ac_power_net'].max()
    capacity_factor = (daily_energy / (SYSTEM_KW_DC * 24)) * 100
    
    print(f"\n📈 Results:")
    print(f"   Daily Energy: {daily_energy/1000:,.2f} KWh")
    print(f"   Peak Power: {peak_power:,.0f} kW")
    print(f"   Capacity Factor: {capacity_factor:.1f}%")
    print(f"   Total Losses: {calculate_total_loss():.1f}%")
    
    # Loss breakdown
    print(f"\n💡 Loss Breakdown:")
    for name, value in LOSSES.items():
        print(f"   {name.capitalize()}: {value:.1f}%")
    
    # Create visualization
    print("\n📊 Creating visualization...")
    fig = create_plot(wx, results, DATE)
    
     #Save to HTML
    output_file = f"solar_forecast_{DATE}.html"
    fig.write_html(output_file, auto_open=True)
    print(f"✅ Saved to: {output_file}")
    
    # Export to CSV
    #export_data = pd.concat([
    #    wx[['ghi', 'dni', 'temp_air', 'wind_speed']],
    #    results[['dc_power', 'ac_power_net', 'cell_temp']]
    #], axis=1)
    #csv_file = f"solar_forecast_{DATE}.csv"
    #export_data.to_csv(csv_file)
    #print(f"📄 Data exported to: {csv_file}")

if __name__ == "__main__":
    main()  

In [ ]:
#%pip install plotly pandas numpy requests pvlib

import pandas as pd
import numpy as np
import requests
from datetime import datetime
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pvlib.location import Location
from pvlib.pvsystem import PVSystem
from pvlib.modelchain import ModelChain
from pvlib.temperature import TEMPERATURE_MODEL_PARAMETERS
from pvlib.irradiance import erbs

# ────────── CONFIG ──────────
LAT, LON = 11.474729, 99.596890
ALTITUDE = 50
SYSTEM_KW_DC = 5000
TILT, AZIMUTH = 15, 180

# Base losses (เปอร์เซ็นต์)
LOSSES = {
    'soiling': 1.0,
    'shading': 1.0,
    'mismatch': 1.0,
    'wiring': 2.0,
    'connections': 1.5,
    'lid': 1.0,
    'nameplate': 1.0,
    'availability': 1.0,
    'degradation': 3
}

def get_weather_open_meteo(day):
    today = datetime.now().date()
    target = datetime.strptime(day, "%Y-%m-%d").date()
    base = "https://archive-api.open-meteo.com/v1/archive" if target < today else "https://api.open-meteo.com/v1/forecast"
    params = ["temperature_2m","cloud_cover","direct_normal_irradiance","diffuse_radiation","wind_speed_10m"]
    url = f"{base}?latitude={LAT}&longitude={LON}&hourly={','.join(params)}&start_date={day}&end_date={day}&timezone=Asia%2FBangkok"
    data = requests.get(url, timeout=20).json()
    h = data["hourly"]
    times = pd.to_datetime(h["time"]).tz_localize("Asia/Bangkok")
    wx = pd.DataFrame({
        'temp_air':    h["temperature_2m"],
        'cloud_cover': h.get("cloud_cover", [0]*len(times)),
        'dni':         h["direct_normal_irradiance"],
        'dhi':         h.get("diffuse_radiation", [0]*len(times)),
        'wind_speed':  h["wind_speed_10m"]
    }, index=times)
    # คำนวณ GHI จาก DNI + DHI
    solpos = Location(LAT, LON).get_solarposition(times)
    zen = solpos['zenith']
    wx['ghi'] = (wx['dni'] * np.cos(np.radians(zen)) + wx['dhi']).clip(lower=0)
    # ถ้า DHI = 0 ทั้งหมด ให้ถอดแบบ ERBS
    if wx['dhi'].sum() == 0:
        decomp = erbs(wx['ghi'], zen, times)
        wx['dhi'], wx['dni'] = decomp['dhi'], decomp['dni']
    return wx

def calculate_dynamic_loss(wx):
    """ปรับ shading loss ตาม avg cloud cover ของวันนั้น"""
    losses = LOSSES.copy()
    avg_cloud = wx['cloud_cover'].mean() / 100.0  # 0–1
    losses['shading'] *= (1 + avg_cloud)         # ถ้าเมฆ 50% → shading = 1*(1+0.5)=1.5%
    eff = np.prod([(1 - v/100) for v in losses.values()])
    return (1 - eff) * 100

def simulate_pv(wx):
    eff_factor = 0.9
    system = PVSystem(
        surface_tilt=TILT,
        surface_azimuth=AZIMUTH,
        module_parameters={'pdc0': SYSTEM_KW_DC * eff_factor, 'gamma_pdc': -0.004},
        inverter_parameters={'pdc0': SYSTEM_KW_DC * eff_factor * 1.05, 'eta_inv_nom': 0.97},
        temperature_model_parameters=TEMPERATURE_MODEL_PARAMETERS['sapm']['open_rack_glass_glass']
    )
    loc = Location(LAT, LON, altitude=ALTITUDE)
    mc = ModelChain(system, loc, dc_model='pvwatts', ac_model='pvwatts', aoi_model='no_loss')
    mc.run_model(wx)

    loss_pct = calculate_dynamic_loss(wx)
    ac_net = (mc.results.ac * (1 - loss_pct/100)).clip(lower=0)

    return pd.DataFrame({
        'dc_power':     mc.results.dc,
        'ac_power_raw': mc.results.ac,
        'ac_power_net': ac_net,
        'cell_temp':    mc.results.cell_temperature,
        'ghi':          wx['ghi'],
        'dni':          wx['dni']
    }, index=wx.index)

def main():
    date_input = input("Enter date (YYYY-MM-DD): ")
    wx = get_weather_open_meteo(date_input)
    res = simulate_pv(wx)

    daily_kwh = res['ac_power_net'].sum() / 1000
    peak_kw   = res['ac_power_net'].max()
    cf        = daily_kwh / (SYSTEM_KW_DC * 24 / 1000) * 100
    loss_pct  = calculate_dynamic_loss(wx)

    fig = make_subplots(rows=2, cols=1, subplot_titles=("AC Power (Net vs Raw)", "Solar Irradiance"), shared_xaxes=True)
    fig.add_trace(go.Scatter(x=res.index, y=res['ac_power_net'], name='AC Net', fill='tozeroy'), 1, 1)
    fig.add_trace(go.Scatter(x=res.index, y=res['ac_power_raw'], name='AC Raw', line=dict(dash='dash')), 1, 1)
    fig.add_trace(go.Scatter(x=res.index, y=res['ghi'], name='GHI'), 2, 1)
    fig.add_trace(go.Scatter(x=res.index, y=res['dni'], name='DNI', line=dict(dash='dot')), 2, 1)

    fig.update_layout(
        title=f"Solar Forecast {date_input} | Daily {daily_kwh:.1f} kWh | Peak {peak_kw:.0f} kW | CF {cf:.1f}% | Loss {loss_pct:.1f}%",
        hovermode="x unified", height=700
    )
    out = f"solar_forecast_{date_input}.html"
    fig.write_html(out, auto_open=True)
    print(f"✅ Saved: {out}")

if __name__ == "__main__":
    main()

In [ ]:
# pip install plotly pandas numpy requests pvlib

import pandas as pd
import numpy as np
import requests
from datetime import datetime, timedelta
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pvlib.location import Location
from pvlib.pvsystem import PVSystem
from pvlib.modelchain import ModelChain
from pvlib.temperature import TEMPERATURE_MODEL_PARAMETERS
from pvlib.irradiance import erbs, get_extra_radiation
import pvlib

# ────────── SYSTEM CONFIGURATION ───────────
LAT, LON = 11.474729262132879, 99.59689042300059
ALTITUDE = 50  # meters
SYSTEM_KW_DC = 5000  # 5 MW
TILT, AZIMUTH = 15, 180
DATE = "2025-07-08"

# System Losses (%)
LOSSES = {
    'soiling': 2.0,
    'shading': 3.0,
    'mismatch': 2.0,
    'wiring': 2.0,
    'connections': 0.5,
    'lid': 1.5,
    'nameplate': 1.0,
    'availability': 2.0,
    'degradation': 0.0  # 0.5% per year * years
}

# ────────── WEATHER DATA FUNCTIONS ───────────
def get_weather_url(day):
    today = datetime.now().date()
    target = datetime.strptime(day, "%Y-%m-%d").date()
    
    if target < today - timedelta(days=1):
        base = "https://historical-forecast-api.open-meteo.com/v1/archive"
        tz   = "&timezone=Asia%2FBangkok"
    else:
    # ยังคงใช้ Forecast API ในปัจจุบัน
        base = "https://api.open-meteo.com/v1/forecast"
        tz   = "&timezone=Asia%2FBangkok"
    
    params = [
        "temperature_2m",
        "relative_humidity_2m",
        "cloud_cover",
        "direct_normal_irradiance",
        "diffuse_radiation",
        "wind_speed_10m"
    ]
    
    return f"{base}?latitude={LAT}&longitude={LON}&hourly={','.join(params)}&start_date={day}&end_date={day}{tz}"

def get_weather(day):
    url = get_weather_url(day)
    resp = requests.get(url, timeout=20)
    data = resp.json()
    
    if "hourly" not in data:
        raise RuntimeError(f"No data for {day}")
    
    h = data["hourly"]
    times = pd.to_datetime(h["time"])
    
    # Timezone handling
    if "archive" in url:
        times = times.tz_localize("UTC").tz_convert("Asia/Bangkok")
    else:
        times = times.tz_localize("Asia/Bangkok")
    
    # Create weather dataframe
    wx = pd.DataFrame({
        'temp_air': h["temperature_2m"],
        'relative_humidity': h.get("relative_humidity_2m", [50]*24),
        'cloud_cover': h.get("cloud_cover", [0]*24),
        'dni': h["direct_normal_irradiance"],
        'dhi': h.get("diffuse_radiation", [0]*24),
        'wind_speed': h["wind_speed_10m"]
    }, index=times)
    
    # Calculate GHI and solar position
    location = Location(LAT, LON, altitude=ALTITUDE)
    solar_pos = location.get_solarposition(times)
    zenith = solar_pos['zenith']
    
    # Calculate GHI
    wx['ghi'] = wx['dni'] * np.cos(np.radians(zenith)) + wx['dhi']
    wx['ghi'] = wx['ghi'].clip(lower=0)
    
    # If DHI missing, use decomposition
    if wx['dhi'].sum() == 0:
        decomp = erbs(wx['ghi'], zenith, times)
        wx['dhi'] = decomp['dhi']
        wx['dni'] = decomp['dni']
    
    return wx

# ────────── PV SYSTEM MODELING ───────────
def calculate_total_loss():
    efficiency = 1.0
    for loss in LOSSES.values():
        efficiency *= (1 - loss/100)
    return (1 - efficiency) * 100

def simulate_pv(wx):
    # Create PV system
    system = PVSystem(
        surface_tilt=TILT,
        surface_azimuth=AZIMUTH,
        module_parameters={
            'pdc0': SYSTEM_KW_DC,
            'gamma_pdc': -0.0045
        },
        inverter_parameters={
            'pdc0': SYSTEM_KW_DC * 1.1,
            'eta_inv_nom': 0.98
        },
        temperature_model_parameters=TEMPERATURE_MODEL_PARAMETERS['sapm']['open_rack_glass_glass']
    )
    
    # Create location and model
    location = Location(LAT, LON, altitude=ALTITUDE)
    mc = ModelChain(
        system, 
        location,
        dc_model='pvwatts',
        ac_model='pvwatts',
        aoi_model='physical',
        spectral_model='no_loss',
        temperature_model='sapm'
    )
    
    # Run simulation
    mc.run_model(wx)
    
    # Apply losses
    total_loss = calculate_total_loss()
    ac_net = mc.results.ac * (1 - total_loss/100)
    
    # Create results
    results = pd.DataFrame({
        'dc_power': mc.results.dc.fillna(0),
        'ac_power_raw': mc.results.ac.fillna(0),
        'ac_power_net': ac_net.fillna(0),
        'cell_temp': mc.results.cell_temperature.fillna(wx['temp_air']),
        'poa_global': mc.results.total_irrad.get('poa_global', pd.Series(0, index=wx.index))
    }, index=wx.index)
    
    return results

# ────────── VISUALIZATION ───────────
def create_plot(wx, results, date):
    # Create figure with subplots
    fig = make_subplots(
        rows=3, cols=1,
        subplot_titles=(
            'Power Output (kW)', 
            'Solar Irradiance (W/m²)', 
            'Temperature (°C)'
        ),
        shared_xaxes=True,
        vertical_spacing=0.08,
        row_heights=[0.5, 0.25, 0.25]
    )
    
    # Time labels
    time_labels = results.index.strftime("%H:%M")
    
    # Plot 1: Power Output
    fig.add_trace(
        go.Scatter(
            x=time_labels,
            y=results['ac_power_net'],
            name='AC Power (Net)',
            line=dict(color='darkblue', width=3),
            fill='tozeroy',
            hovertemplate='Net Power: %{y:,.0f} kW<extra></extra>'
        ), 
        row=1, col=1
    )
    
    fig.add_trace(
        go.Scatter(
            x=time_labels,
            y=results['ac_power_raw'],
            name='AC Power (Before Losses)',
            line=dict(color='lightblue', dash='dash'),
            hovertemplate='Raw Power: %{y:,.0f} kW<extra></extra>'
        ), 
        row=1, col=1
    )
    
    # Plot 2: Irradiance
    fig.add_trace(
        go.Scatter(
            x=time_labels,
            y=wx['ghi'],
            name='GHI',
            line=dict(color='orange'),
            hovertemplate='GHI: %{y:.0f} W/m²<extra></extra>'
        ), 
        row=2, col=1
    )
    
    fig.add_trace(
        go.Scatter(
            x=time_labels,
            y=wx['dni'],
            name='DNI',
            line=dict(color='red'),
            hovertemplate='DNI: %{y:.0f} W/m²<extra></extra>'
        ), 
        row=2, col=1
    )
    
    # Plot 3: Temperature
    fig.add_trace(
        go.Scatter(
            x=time_labels,
            y=wx['temp_air'],
            name='Air Temp',
            line=dict(color='green'),
            hovertemplate='Air: %{y:.1f}°C<extra></extra>'
        ), 
        row=3, col=1
    )
    
    fig.add_trace(
        go.Scatter(
            x=time_labels,
            y=results['cell_temp'],
            name='Cell Temp',
            line=dict(color='darkgreen', dash='dash'),
            hovertemplate='Cell: %{y:.1f}°C<extra></extra>'
        ), 
        row=3, col=1
    )
    
    # Update layout
    total_loss = calculate_total_loss()
    daily_energy = results['ac_power_net'].sum()
    peak_power = results['ac_power_net'].max()
    capacity_factor = (daily_energy / (SYSTEM_KW_DC * 24)) * 100
    
    fig.update_layout(
        title={
            'text': f"Solar Power Forecast - {SYSTEM_KW_DC/1000:.1f} MW System<br>" + 
                   f"<span style='font-size:14px'>Date: {date} | " +
                   f"Daily Energy: {daily_energy/1000:,.1f} KWh | " +
                   f"Peak: {peak_power:,.0f} kW | " +
                   f"CF: {capacity_factor:.1f}% | " +
                   f"Losses: {total_loss:.1f}%</span>",
            'x': 0.5,
            'xanchor': 'center'
        },
        xaxis3_title="Time (Hour)",
        yaxis_title="Power (kW)",
        yaxis2_title="Irradiance (W/m²)",
        yaxis3_title="Temperature (°C)",
        height=800,
        showlegend=True,
        hovermode='x unified'
    )
    
    # Update axes
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
    
    return fig

# ────────── MAIN EXECUTION ───────────
def main():
    print(f"🌞 Solar Forecast for {DATE}")
    print(f"📍 Location: {LAT:.4f}, {LON:.4f}")
    print(f"⚡ System Size: {SYSTEM_KW_DC/1000:.1f} MW")
    
    # Get weather data
    print("\n📊 Fetching weather data...")
    wx = get_weather(DATE)
    
    # Run PV simulation
    print("🔧 Running PV simulation...")
    results = simulate_pv(wx)
    
    # Calculate metrics
    daily_energy = results['ac_power_net'].sum()
    peak_power = results['ac_power_net'].max()
    capacity_factor = (daily_energy / (SYSTEM_KW_DC * 24)) * 100
    
    print(f"\n📈 Results:")
    print(f"   Daily Energy: {daily_energy/1000:,.2f} KWh")
    print(f"   Peak Power: {peak_power:,.0f} kW")
    print(f"   Capacity Factor: {capacity_factor:.1f}%")
    print(f"   Total Losses: {calculate_total_loss():.1f}%")
    
    # Loss breakdown
    print(f"\n💡 Loss Breakdown:")
    for name, value in LOSSES.items():
        print(f"   {name.capitalize()}: {value:.1f}%")
    
    # Create visualization
    print("\n📊 Creating visualization...")
    fig = create_plot(wx, results, DATE)
    
    # Save to HTML
    output_file = f"solar_forecast_{DATE}.html"
    fig.write_html(output_file, auto_open=True)
    print(f"✅ Saved to: {output_file}")
    
    # Export to CSV
    export_data = pd.concat([
        wx[['ghi', 'dni', 'temp_air', 'wind_speed']],
        results[['dc_power', 'ac_power_net', 'cell_temp']]
    ], axis=1)
    csv_file = f"solar_forecast_{DATE}.csv"
    export_data.to_csv(csv_file)
    print(f"📄 Data exported to: {csv_file}")

if __name__ == "__main__":
    main()

In [ ]:
# pip install plotly pandas numpy requests pvlib

import pandas as pd
import numpy as np
import requests
from datetime import datetime, timedelta
import plotly.graph_objects as go

from pvlib.location import Location
from pvlib.pvsystem import PVSystem
from pvlib.modelchain import ModelChain
import pvlib

# ────────── SETTINGS ───────────
LAT, LON = 11.474729262132879, 99.59689042300059
SYSTEM_KW_DC = 5000
TILT, AZIMUTH = 15, 180
DATE = "2025-07-14"  # 🔁 เปลี่ยนวันย้อนหลัง / ล่วงหน้า ได้เลย

# ────────── Helper Functions ───────────
def build_url(day):
    today = datetime.now().date()
    tgt = datetime.strptime(day, "%Y-%m-%d").date()
    if tgt < today - timedelta(days=1):
        base, tz = "https://archive-api.open-meteo.com/v1/archive", ""
    else:
        base, tz = "https://api.open-meteo.com/v1/forecast", "&timezone=Asia%2FBangkok"
    return (
        f"{base}?latitude={LAT}&longitude={LON}"
        "&hourly=temperature_2m,direct_normal_irradiance,wind_speed_10m"
        f"&start_date={day}&end_date={day}{tz}"
    )

def get_weather(day: str) -> pd.DataFrame:
    url = build_url(day)
    js = requests.get(url, timeout=20).json()
    if "hourly" not in js:
        raise RuntimeError(f"❌ ไม่พบข้อมูล {day}: {js}")
    h = js["hourly"]
    t = pd.to_datetime(h["time"])
    t = (t.tz_localize("Asia/Bangkok")
         if "/forecast" in url
         else t.tz_localize("UTC").tz_convert("Asia/Bangkok"))

    temp = pd.Series(h["temperature_2m"], index=t, name="temp")
    wind = pd.Series(h["wind_speed_10m"], index=t, name="wind")
    dni = pd.Series(h["direct_normal_irradiance"], index=t, name="dni")

    zen = Location(LAT, LON).get_solarposition(t)["zenith"]
    cosz = np.cos(np.radians(zen))
    ghi = (dni * cosz).clip(lower=0)
    dhi = (ghi - dni * cosz).clip(lower=0)

    df = pd.concat({"ghi": ghi, "dni": dni, "dhi": dhi,
                    "temp_air": temp, "wind_speed": wind}, axis=1)
    
    full_hours = pd.date_range(f"{day} 00:00", f"{day} 23:00", freq="1H", tz="Asia/Bangkok")
    return df.reindex(full_hours).fillna(0)

def get_ac_power(wx: pd.DataFrame) -> pd.Series:
    system = PVSystem(
        surface_tilt=TILT, surface_azimuth=AZIMUTH,
        module_parameters={"pdc0": SYSTEM_KW_DC, "gamma_pdc": -0.003},
        inverter_parameters={"pdc0": SYSTEM_KW_DC},
        temperature_model_parameters=pvlib.temperature.TEMPERATURE_MODEL_PARAMETERS["sapm"]["open_rack_glass_glass"]
    )
    mc = ModelChain(system, Location(LAT, LON),
                    dc_model="pvwatts", ac_model="pvwatts", aoi_model="no_loss")
    mc.run_model(wx)
    return mc.results.ac.reindex(wx.index).fillna(0)

# ────────── Run Calculation ───────────
wx = get_weather(DATE)
ac = get_ac_power(wx)

# ────────── Plotly Interactive Chart ───────────
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=ac.index.strftime("%H:%M"),
    y=ac.values,
    mode="lines+markers",
    name="Solar output (KWH)",
    line=dict(color="royalblue")
))

fig.update_layout(
    title=f"Solar – {SYSTEM_KW_DC / 1000:.0f} MW<br>{DATE}",
    xaxis_title="Time ",
    yaxis_title="Solar output (KWH)",
    legend=dict(x=0.01, y=0.99),
    height=500
)

fig.write_html("ac_power_only.html", auto_open=True)

In [ ]:
# ───────────── Imports ──────────────
from datetime import datetime, timedelta
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

import pvlib
from pvlib.location import Location
from pvlib.pvsystem import PVSystem
from pvlib.modelchain import ModelChain

# ────────── USER SETTINGS ───────────
LAT, LON      = 11.474729262132879, 99.59689042300059  # Lopburi Absolute Solar farm
DATE          = "2025-07-02"                           # วันที่ต้องการ (YYYY-MM-DD)
SYSTEM_KW_DC  = 55_000                                 # ขนาดติดตั้ง (kW)
TILT, AZIMUTH = 15, 180                                # แผงเอียง 15° หันใต้
# ────────────────────────────────────

# ---------- 1) สร้าง URL สำหรับดึงข้อมูล ----------
def build_url(day: str) -> str:
    today = datetime.now().date()
    tgt = datetime.strptime(day, "%Y-%m-%d").date()

    if tgt < today - timedelta(days=1):  # อดีต → archive (UTC)
        base, tz = "https://archive-api.open-meteo.com/v1/archive", ""
    else:                                # วันนี้หรืออนาคต ≤ 16 วัน → forecast
        base, tz = "https://api.open-meteo.com/v1/forecast", "&timezone=Asia%2FBangkok"

    return (
        f"{base}?latitude={LAT}&longitude={LON}"
        "&hourly=temperature_2m,direct_normal_irradiance,wind_speed_10m"
        f"&start_date={day}&end_date={day}{tz}"
    )

# ---------- 2) ดึงข้อมูลสภาพอากาศ ----------
def get_weather(day: str) -> pd.DataFrame:
    url = build_url(day)
    response = requests.get(url, timeout=20)
    data = response.json()

    if "hourly" not in data:
        raise RuntimeError(f"⚠️ ไม่พบข้อมูล hourly: {data}")

    h = data["hourly"]
    t = pd.to_datetime(h["time"])
    t = (t.tz_localize("Asia/Bangkok")
         if "/forecast" in url
         else t.tz_localize("UTC").tz_convert("Asia/Bangkok"))

    dni = pd.Series(h["direct_normal_irradiance"], index=t, name="dni")
    temp = pd.Series(h["temperature_2m"], index=t, name="temp")
    wind = pd.Series(h["wind_speed_10m"], index=t, name="wind")

    zen = Location(LAT, LON).get_solarposition(t)["zenith"]
    cosz = np.cos(np.radians(zen))
    ghi = (dni * cosz).clip(lower=0)
    dhi = (ghi - dni * cosz).clip(lower=0)

    df = pd.concat({"ghi": ghi, "dni": dni, "dhi": dhi,
                    "temp_air": temp, "wind_speed": wind}, axis=1)

    # เติมค่า 0 ให้ชั่วโมงที่ไม่มีข้อมูล (00:00–23:00)
    full_hours = pd.date_range(f"{day} 00:00", f"{day} 23:00",
                               freq="1H", tz="Asia/Bangkok")
    return df.reindex(full_hours).fillna(0)

# ---------- 3) คำนวณกำลังไฟฟ้า AC ----------
def pv_power_ac(weather: pd.DataFrame) -> pd.Series:
    system = PVSystem(
        surface_tilt=TILT,
        surface_azimuth=AZIMUTH,
        module_parameters={"pdc0": SYSTEM_KW_DC, "gamma_pdc": -0.003},
        inverter_parameters={"pdc0": SYSTEM_KW_DC},
        temperature_model_parameters=pvlib.temperature.TEMPERATURE_MODEL_PARAMETERS["sapm"]["open_rack_glass_glass"]
    )
    mc = ModelChain(system, Location(LAT, LON),
                    dc_model="pvwatts", ac_model="pvwatts", aoi_model="no_loss")
    mc.run_model(weather)
    return mc.results.ac.reindex(weather.index).fillna(0)

# ---------- 4) วาดกราฟ ----------
def plot_day():
    wx = get_weather(DATE)
    ac = pv_power_ac(wx)

    fig, ax1 = plt.subplots(figsize=(10, 5))
    ac.plot(ax=ax1, label="AC Power (W)", color="tab:blue")
    ax1.set_ylabel("AC Power (W)", color="tab:blue")

    ax2 = ax1.twinx()
    wx["dni"].plot(ax=ax2, linestyle="--", label="DNI (W/m²)", color="tab:orange")
    ax2.set_ylabel("DNI (W/m²)", color="tab:orange")
    ax2.tick_params(axis="x", labelbottom=False)

    # ตั้งค่าแกน X: tick ทุก 3 ชั่วโมง
    ticks = pd.date_range(f"{DATE} 00:00", f"{DATE} 21:00", freq="3H", tz="Asia/Bangkok")
    ax1.set_xlim(wx.index[0], wx.index[-1])
    ax1.set_xticks(ticks)
    ax1.set_xticklabels([ts.strftime("%H:%M") for ts in ticks], rotation=45, ha="right", fontsize=9)
    ax1.xaxis.set_minor_locator(mticker.NullLocator())

    plt.title(f"Solar – {SYSTEM_KW_DC / 1000:.0f} MW\n{DATE}")
    fig.legend(loc="upper right")
    plt.grid(True, linestyle=":", axis="both")
    plt.tight_layout()
    plt.show()

# -------------- RUN --------------
if __name__ == "__main__":
    plot_day()

In [ ]:
# ติดตั้งไลบรารีก่อน
%pip install plotly pandas

import pandas as pd
import plotly.graph_objects as go

# โหลดข้อมูล
df = pd.read_csv("TSK_PV-160768.csv")
df['TIME_LOCAL'] = pd.to_datetime(df['TIME_LOCAL'], format="%d/%m/%Y %H:%M:%S")
df['Date'] = df['TIME_LOCAL'].dt.date.astype(str)

# เตรียม figure
fig = go.Figure()

# สร้างเส้นกราฟแยกตามวัน และซ่อนทุกเส้นไว้ก่อน (ยกเว้นเส้นแรก)
unique_dates = df['Date'].unique()

for i, date in enumerate(unique_dates):
    filtered = df[df['Date'] == date]
    visible = True if i == 0 else False  # แสดงเฉพาะเส้นแรก
    fig.add_trace(go.Scatter(
        x=filtered['TIME_LOCAL'],
        y=filtered['Solar'],
        mode='lines+markers',
        name=str(date),
        visible=visible
    ))

# สร้าง dropdown สำหรับเลือกวัน
dropdown_buttons = [
    {
        "label": date,
        "method": "update",
        "args": [
            {"visible": [d == date for d in unique_dates]},
            {"title": f"Solar Output on {date}"}
        ]
    } for date in unique_dates
]

# ปรับ layout
fig.update_layout(
    updatemenus=[{
        "buttons": dropdown_buttons,
        "direction": "down",
        "showactive": True,
        "x": 1.1,
        "xanchor": "left",
        "y": 1.1,
        "yanchor": "top"
    }],
    title=f"Solar Output on {unique_dates[0]}",
    xaxis_title="Time",
    yaxis_title="Solar Output (kWh)"
)

# แสดงผล
fig.write_html("solar_dropdown.html", auto_open=True)